# CE541E08 — Unit 5 · Day 40 — Colon Notation, Element-wise Ops and Array Functions
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 5 — Introduction to MATLAB |
| **Session** | Day 40 of 45 |
| **Topics** | colon · linspace · .* and ./ · array functions · AMS · nested loops |
---
> **Copy each MATLAB code block and run it in MATLAB Online** at [matlab.mathworks.com](https://matlab.mathworks.com).
> Read the explanation and algorithm first. Verify your output matches the expected output.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 40"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Vectorised Operations in MATLAB

In MATLAB, `.` before an operator means **element-wise**: `.*`, `./`, `.^`. Without the dot, `*` is matrix multiplication, `/` is matrix division — very different operations.

| Operator | MATLAB | NumPy equivalent |
|---|---|---|
| Element-wise multiply | `A .* B` | `A * B` |
| Element-wise divide | `A ./ B` | `A / B` |
| Element-wise power | `A .^ 2` | `A ** 2` |
| Matrix multiply | `A * B` | `A @ B` |

For arrays with the same size, always use `.*`, `./`, `.^` in civil engineering calculations — matrix multiplication is only needed for structural stiffness equations.

---
## Code Block 1 — Colon and linspace

### What this code does

We create engineering sequences using the colon operator and `linspace`, then verify their lengths and values.

### Why each step is taken

**`1:30`:**
Creates the integer sequence [1, 2, ..., 30]. Used as day numbers, pipe section indices, or year labels.

**`0.001:0.001:0.01`:**
Creates [0.001, 0.002, ..., 0.010] — 10 slope values for a parametric study. The general form is `start:step:stop`.

**`150:50:600`:**
Pipe diameters [150, 200, 250, ..., 600] mm.

**`linspace(20, 120, 11)`:**
Generates exactly 11 equally spaced values between 20 and 120. Unlike the colon operator where you specify the step, `linspace` lets you specify the count. This is `np.linspace` in Python.

### Algorithm

```
1. days   = 1:30         → [1,2,...,30]
2. slopes = 0.001:0.001:0.01 → [0.001,...,0.01]
3. D_mm   = 150:50:600   → [150,200,...,600]
4. P_storms = linspace(20,120,11) → 11 values
5. Print lengths and values
```

### Expected output

```
Days: 30 values
Slopes: 10 values from 0.001 to 0.010
Diameters: 10 values
Storm depths (mm): 20 30 40 50 60 70 80 90 100 110 120
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% CE541E08 - Day 40: Colon Notation
days     = 1:30;
slopes   = 0.001:0.001:0.01;
D_mm     = 150:50:600;
P_storms = linspace(20,120,11);

fprintf('Days: %d values\n', length(days))
fprintf('Slopes: %d values from %.3f to %.3f\n', length(slopes), min(slopes), max(slopes))
fprintf('Diameters: %d values\n', length(D_mm))
fprintf('Storm depths (mm): '); disp(P_storms)
""")

### 🔁 Try this

Create a vector of 20 AMS values from 1000 to 5000 m³/s using `linspace(1000,5000,20)`.

- Use `mean()`, `std()`, `max()` on the result
- Confirm that the spacing between consecutive values is constant using `diff(v)` — all values should be equal

---
## Code Block 2 — Element-wise: Manning's for Multiple Diameters

### What this code does

We compute Manning's velocity and discharge for 5 pipe diameters simultaneously using element-wise operators — no loop.

### Why each step is taken

**`D = D_mm/1000`:**
Scalar division applied to a vector. MATLAB divides every element by 1000 automatically. No `.` needed when one operand is a scalar.

**`R = D/4`:**
Again scalar — every element of D divided by 4.

**`A_pipe = pi .* (D/2).^2`:**
Here we need `.^` because D is a vector. `(D/2)^2` would attempt matrix exponentiation — wrong. `(D/2).^2` applies the square element-wise.

**`V = (1/n) .* R.^(2/3) .* S^0.5`:**
`R.^(2/3)` applies the 2/3 power element-wise. `S^0.5` is a scalar (S is a single value). The result V is a vector of 5 velocities.

### Algorithm

```
1. D_mm = [200,250,300,350,400]  → vector
   D = D_mm/1000                  → scalar division, still vector
   S = 0.002 (scalar)
   n = 0.013 (scalar)

2. R = D/4                    (scalar/vector = vector)
   A = pi .* (D/2).^2         (element-wise — D is vector)
   V = (1/n) .* R.^(2/3) .* S^0.5  (element-wise)
   Q = V .* A                  (element-wise multiply)

3. Print table: D(mm), V(m/s), Q(L/s)
```

### Expected output

```
D(mm)  V(m/s)   Q(L/s)
  200   0.921     28.84
  250   1.073     52.61
  300   1.213     85.61
  350   1.343    128.94
  400   1.466    183.73
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% Element-wise operations (.* and ./)
D_mm=[200,250,300,350,400]; D=D_mm/1000;
S=0.002; n=0.013;
R=D/4;
A_pipe=pi.*(D/2).^2;          % .^ needed: D is a vector
V=(1/n).*R.^(2/3).*S^0.5;    % .^ and .* for element-wise
Q=V.*A_pipe;

fprintf('%-8s %-10s %-10s\n','D(mm)','V(m/s)','Q(L/s)')
for i=1:length(D_mm)
    fprintf('%-8d %-10.3f %-10.2f\n',D_mm(i),V(i),Q(i)*1000)
end
""")

### 🔁 Try this

Add a velocity check using a vectorised condition (no loop):

`ok = (V >= 0.6) & (V <= 3.0);`

`fprintf('Acceptable: %d of %d\n', sum(ok), length(ok))`

The `&` is element-wise AND for vectors in MATLAB.

---
## Code Block 3 — AMS Statistics with Array Functions

### What this code does

We compute descriptive statistics of a 20-year Annual Maximum Series using MATLAB's built-in array functions, then build the Weibull flood frequency table.

### Why each step is taken

**`mean(AMS)`, `std(AMS)`, `max(AMS)`, `median(AMS)`:**
All built-in MATLAB functions that operate on the entire array at once. No axis argument needed for 1-D vectors.

**`sort(AMS,'descend')`:**
Sorts the AMS from largest to smallest. The second argument `'descend'` specifies the direction. In Python/NumPy: `np.sort(AMS)[::-1]`.

**`P = (1:n)/(n+1)`:**
Weibull plotting position formula applied to the entire rank vector at once. `(1:n)` creates [1,2,...,n]; dividing by `(n+1)` applies element-wise. No loop needed.

**`T = 1./P`:**
Element-wise division: `1` divided by each P value. Must use `./` because P is a vector.

### Algorithm

```
1. AMS = 20-year peak flow array

2. mean, std, max, median — direct function calls

3. AMS_s = sort(AMS,'descend')  → sorted largest to smallest
   n = length(AMS_s)
   P = (1:n)/(n+1)              → exceedance probabilities
   T = 1./P                     → return periods

4. Print top 5 events with return periods
```

### Expected output

```
n      : 20 years
Mean   : 3024.0 m3/s
Std    : 1114.7 m3/s
Max    : 5234.0 m3/s
Median : 2890.0 m3/s
Top 5 and return periods:
Rank 1: 5234 m3/s  T=26.0 yr
...
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% AMS analysis using array functions
AMS=[1234,2456,1890,3456,2234,4567,3123,1567,2890,5234,...
     3456,2123,4567,2890,1678,3234,4890,2345,3678,1456];
fprintf('n      : %d years\n', length(AMS))
fprintf('Mean   : %.1f m3/s\n', mean(AMS))
fprintf('Std    : %.1f m3/s\n', std(AMS))
fprintf('Max    : %.1f m3/s\n', max(AMS))
fprintf('Median : %.1f m3/s\n', median(AMS))

AMS_s = sort(AMS,'descend');
n     = length(AMS_s);
P     = (1:n)/(n+1);   % Weibull: rank/(n+1) for all ranks at once
T     = 1./P;           % element-wise: 1/P for each value

fprintf('Top 5 and return periods:\n')
for i = 1:5
    fprintf('Rank %d: %.0f m3/s  T=%.1f yr\n', i, AMS_s(i), T(i))
end
""")

### 🔁 Try this

Find the approximate 10-year flood using `interp1`:

`Q10 = interp1(P, AMS_s, 1/10, 'linear');`
`fprintf('T=10 yr flood: %.0f m3/s\n', Q10)`

`interp1` is MATLAB's 1-D interpolation function — equivalent to `np.interp`.

---
## Code Block 4 — Nested Loops: Full Sizing Table

### What this code does

We generate the complete Manning's pipe sizing table using a nested for loop over diameters and slopes — producing Q(L/s) for every combination.

### Why each step is taken

**`D_mm_range = 150:50:400`:**
Diameters from 150 to 400 mm in steps of 50 mm. Using the colon operator means changing the range requires editing only one line.

**`S_range = [0.001, 0.002, 0.003, 0.005]`:**
A manually specified vector of non-uniform slopes. The inner for loop iterates over each value.

**Two nested loops:**
The outer loop steps through diameters (rows), the inner loop steps through slopes (columns). The outer loop prints the diameter at the start of each row and a newline at the end. The inner loop prints each Q value without a newline.

### Algorithm

```
1. D_mm_range = 150:50:400  (6 values)
   S_range = [0.001,...,0.005]  (4 values)

2. Print header row (slope labels)

3. For each D in D_mm_range:   % outer loop — rows
   Print D
   For each S in S_range:      % inner loop — columns
     Compute V and Q
     Print Q (no newline)
   Print newline

4. Total: 6 × 4 = 24 Q values
```

### Expected output

```
D(mm) S=1:1000  S=1:500   S=1:333   S=1:200
  150     30.62     43.30     53.02     68.51
  200     61.17     86.49    105.88    136.82
  ...
  400    315.46    446.15    546.31    705.88
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% Nested loop: diameter-slope sizing table
D_mm_range = 150:50:400;
S_range    = [0.001,0.002,0.003,0.005];
n          = 0.013;

fprintf('%6s','D(mm)');
for S = S_range
    fprintf(' S=1:%-5d', round(1/S));
end
fprintf('\n');

for D_mm = D_mm_range
    D=D_mm/1000; R=D/4; A=pi*(D/2)^2;
    fprintf('%6d', D_mm);
    for S = S_range
        V=(1/n)*R^(2/3)*S^0.5; Q=V*A;
        fprintf(' %8.2f', Q*1000);
    end
    fprintf('\n')
end
""")

### 🔁 Try this

Extend the diameter range to `150:50:600` (10 diameters).

- How many Q values does the table now have?
- Which combination gives flow closest to 100 L/s?

---
## Session Summary — Colon and Element-wise

| Concept | MATLAB | Python/NumPy |
|---|---|---|
| Integer sequence | `1:n` | `np.arange(1,n+1)` |
| Stepped sequence | `a:step:b` | `np.arange(a,b+step,step)` |
| n equally spaced | `linspace(a,b,n)` | `np.linspace(a,b,n)` |
| Element-wise * | `A .* B` | `A * B` |
| Element-wise / | `A ./ B` | `A / B` |
| Element-wise ^ | `A .^ p` | `A ** p` |
| Sort descending | `sort(v,'descend')` | `np.sort(v)[::-1]` |
| 1-D interpolation | `interp1(x,y,xi)` | `np.interp(xi,x,y)` |
| For over vector | `for x = v_name` | `for x in v_name:` |

---
## Day 40 Assignment

Using element-wise operations (no loop for the main calculation), compute SCS-CN runoff Q (mm) and volume (Mm³) for the 6 Cauvery catchments for a P=85mm storm. Then use a for loop only for printing the formatted table.

In [ ]:
# Copy and run this in MATLAB Online
print("""
% Day40_Assignment.m - SCS-CN vectorised
clc; clear;
areas = [2950, 1930, 7040, 1575, 4762, 3101];  % km2
CN    = [72, 68, 75, 70, 65, 73];
P     = 85;   % storm depth, mm

% Vectorised (no loop for computation)
S  = 25400./CN - 254;
Ia = 0.2.*S;
Q  = zeros(size(CN));
for i = 1:length(CN)
    if P > Ia(i), Q(i) = (P-Ia(i))^2/(P-Ia(i)+S(i)); end
end
Vol = Q./1000.*areas;

% Print table
names = {'Hemavathi','Harangi','Kabini','Suvarnavathi','Shimsha','Arkavathi'};
fprintf('%-15s %6s %8s %10s\n','Catchment','CN','Q(mm)','Vol(Mm3)')
for i=1:length(CN)
    fprintf('%-15s %6d %8.2f %10.3f\n',names{i},CN(i),Q(i),Vol(i))
end
fprintf('%-15s %6s %8s %10.3f\n','TOTAL','','',sum(Vol))
""")

---
- [ ] Run all MATLAB blocks in MATLAB Online — verify outputs
- [ ] Save scripts as `.m` files
- [ ] Upload: `Unit5_MATLAB/CE541E08_U5_Day40.ipynb`
- [ ] Commit: `Day 40 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*